In [1]:
# ============================================================
# 03_mouse_encoders.ipynb
# Mouse Dynamics — Session Encoders (MLP classifier + AE)
# ============================================================

"""
Goal:
  - Load mouse session-level features from mouse_features.npz
  - Train:
      (1) A small MLP classifier that predicts user_id
          -> Use its embedding layer as a discriminative representation
      (2) A small autoencoder that reconstructs the features
          -> Unsupervised embedding capturing typical behavior
  - Export embeddings for all sessions:
      mouse_embeddings_cls.npy  (discriminative)
      mouse_embeddings_ae.npy   (unsupervised)
"""

# ============================================================
# Setup
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

# Paths
DATA_DIR = Path("data")
MODEL_DIR = Path("models")
CKPT_DIR = Path("checkpoints")
EMB_DIR = Path("embeddings")

MODEL_DIR.mkdir(exist_ok=True)
CKPT_DIR.mkdir(exist_ok=True)
EMB_DIR.mkdir(exist_ok=True)

FEATURES_PATH = DATA_DIR / "mouse_features.npz"

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [2]:
# ============================================================
# 1. Load features
# ============================================================

data = np.load(FEATURES_PATH, allow_pickle=True)
X = data["features"].astype(np.float32)      # (N_sessions, 33)
user_ids = data["user_id"].astype(int)       # (N_sessions,)
session_ids = data["session_id"]             # (N_sessions,)

N, D = X.shape
unique_users = np.unique(user_ids)
n_users = unique_users.shape[0]

print("Feature matrix:", X.shape)
print("Unique users:", n_users)
print("Users:", unique_users)

# Map user_ids to 0..n_users-1
user_to_idx = {u: i for i, u in enumerate(sorted(unique_users))}
y = np.array([user_to_idx[u] for u in user_ids], dtype=np.int64)

# Quick per-user counts
print("\nSessions per user:")
print(pd.Series(user_ids).value_counts().sort_index())

Feature matrix: (1676, 33)
Unique users: 10
Users: [ 7  9 12 15 16 20 21 23 29 35]

Sessions per user:
7     158
9     130
12    240
15    253
16    201
20    114
21    121
23    142
29    124
35    193
Name: count, dtype: int64


In [3]:
# ============================================================
# 2. Train/val split + scaling
# ============================================================

X_train, X_val, y_train, y_val, u_train, u_val, s_train, s_val = train_test_split(
    X, y, user_ids, session_ids,
    test_size=0.2,
    random_state=SEED,
    stratify=y,  # balanced across users
)

print("Train:", X_train.shape, "Val:", X_val.shape)

# Standardize features (fit on train, apply to all)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_all_scaled = scaler.transform(X)

SCALER_PATH = MODEL_DIR / "mouse_scaler.joblib"
joblib.dump(scaler, SCALER_PATH)
print("💾 Saved scaler →", SCALER_PATH)

Train: (1340, 33) Val: (336, 33)
💾 Saved scaler → models/mouse_scaler.joblib


In [4]:
# ============================================================
# 3. Torch Dataset / DataLoader
# ============================================================

class MouseSessionDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.from_numpy(X.astype(np.float32))
        self.y = None if y is None else torch.from_numpy(y.astype(np.int64))

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx]
        return self.X[idx], self.y[idx]

train_ds = MouseSessionDataset(X_train_scaled, y_train)
val_ds   = MouseSessionDataset(X_val_scaled, y_val)
all_ds   = MouseSessionDataset(X_all_scaled, y)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False)
all_loader   = DataLoader(all_ds, batch_size=64, shuffle=False)

In [5]:
# ============================================================
# 4. MLP classifier encoder
# ============================================================

class MLPClassifierEncoder(nn.Module):
    def __init__(self, in_dim, emb_dim, n_classes):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.ReLU(),
            nn.Linear(64, emb_dim),
            nn.ReLU(),
        )
        self.class_head = nn.Linear(emb_dim, n_classes)

    def forward(self, x):
        z = self.encoder(x)
        logits = self.class_head(z)
        return logits

    def encode(self, x):
        with torch.no_grad():
            return self.encoder(x)

in_dim = D
emb_dim_cls = 32

clf = MLPClassifierEncoder(in_dim, emb_dim_cls, n_users).to(device)
criterion_ce = nn.CrossEntropyLoss()
optimizer_clf = torch.optim.Adam(clf.parameters(), lr=1e-3, weight_decay=1e-4)

print(clf)

MLPClassifierEncoder(
  (encoder): Sequential(
    (0): Linear(in_features=33, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
  )
  (class_head): Linear(in_features=32, out_features=10, bias=True)
)


In [6]:
# ============================================================
# 5. Train classifier
# ============================================================

def eval_classifier(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return correct / max(total, 1)

EPOCHS_CLF = 30
best_val_acc = 0.0
best_clf_path = CKPT_DIR / "mouse_mlp_classifier.pt"

print("\n🔧 Training MLP classifier encoder...")
for epoch in range(1, EPOCHS_CLF + 1):
    clf.train()
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer_clf.zero_grad()
        logits = clf(xb)
        loss = criterion_ce(logits, yb)
        loss.backward()
        optimizer_clf.step()

    if epoch % 2 == 0 or epoch == 1:
        train_acc = eval_classifier(clf, train_loader)
        val_acc = eval_classifier(clf, val_loader)
        print(f"[CLS] epoch {epoch:02d} train_acc={train_acc:.4f} val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(clf.state_dict(), best_clf_path)

print(f"✅ Best classifier val_acc={best_val_acc:.4f}")
print("💾 Saved classifier checkpoint →", best_clf_path)


🔧 Training MLP classifier encoder...
[CLS] epoch 01 train_acc=0.2015 val_acc=0.1815
[CLS] epoch 02 train_acc=0.2582 val_acc=0.2560
[CLS] epoch 04 train_acc=0.3388 val_acc=0.3095
[CLS] epoch 06 train_acc=0.3552 val_acc=0.3155
[CLS] epoch 08 train_acc=0.3664 val_acc=0.3155
[CLS] epoch 10 train_acc=0.3754 val_acc=0.3244
[CLS] epoch 12 train_acc=0.3910 val_acc=0.3304
[CLS] epoch 14 train_acc=0.4060 val_acc=0.3333
[CLS] epoch 16 train_acc=0.4187 val_acc=0.3571
[CLS] epoch 18 train_acc=0.4239 val_acc=0.3482
[CLS] epoch 20 train_acc=0.4366 val_acc=0.3571
[CLS] epoch 22 train_acc=0.4396 val_acc=0.3571
[CLS] epoch 24 train_acc=0.4425 val_acc=0.3661
[CLS] epoch 26 train_acc=0.4485 val_acc=0.3720
[CLS] epoch 28 train_acc=0.4552 val_acc=0.3780
[CLS] epoch 30 train_acc=0.4664 val_acc=0.3780
✅ Best classifier val_acc=0.3780
💾 Saved classifier checkpoint → checkpoints/mouse_mlp_classifier.pt


In [7]:
# ============================================================
# 6. Autoencoder
# ============================================================

class MouseAutoencoder(nn.Module):
    def __init__(self, in_dim, emb_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.ReLU(),
            nn.Linear(64, emb_dim),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(emb_dim, 64),
            nn.ReLU(),
            nn.Linear(64, in_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        recon = self.decoder(z)
        return recon

    def encode(self, x):
        with torch.no_grad():
            return self.encoder(x)

emb_dim_ae = 16
ae = MouseAutoencoder(in_dim, emb_dim_ae).to(device)
criterion_mse = nn.MSELoss()
optimizer_ae = torch.optim.Adam(ae.parameters(), lr=1e-3, weight_decay=1e-5)

print(ae)

MouseAutoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=33, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=16, bias=True)
    (3): ReLU()
  )
  (decoder): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=33, bias=True)
  )
)


In [8]:
# ============================================================
# 7. Train autoencoder
# ============================================================

EPOCHS_AE = 60
best_recon = float("inf")
best_ae_path = CKPT_DIR / "mouse_autoencoder.pt"

print("\n🔧 Training autoencoder...")
for epoch in range(1, EPOCHS_AE + 1):
    ae.train()
    epoch_loss = 0.0
    n_batches = 0

    for xb, _ in train_loader:
        xb = xb.to(device)

        optimizer_ae.zero_grad()
        recon = ae(xb)
        loss = criterion_mse(recon, xb)
        loss.backward()
        optimizer_ae.step()

        epoch_loss += loss.item()
        n_batches += 1

    recon_loss = epoch_loss / max(n_batches, 1)
    if epoch % 5 == 0 or epoch == 1:
        print(f"[AE ] epoch {epoch:02d} recon_loss={recon_loss:.6f}")

    if recon_loss < best_recon:
        best_recon = recon_loss
        torch.save(ae.state_dict(), best_ae_path)

print(f"✅ Best AE recon_loss={best_recon:.6f}")
print("💾 Saved AE checkpoint →", best_ae_path)


🔧 Training autoencoder...
[AE ] epoch 01 recon_loss=0.855895
[AE ] epoch 05 recon_loss=0.401486
[AE ] epoch 10 recon_loss=0.220388
[AE ] epoch 15 recon_loss=0.133219
[AE ] epoch 20 recon_loss=0.088609
[AE ] epoch 25 recon_loss=0.063707
[AE ] epoch 30 recon_loss=0.046748
[AE ] epoch 35 recon_loss=0.035072
[AE ] epoch 40 recon_loss=0.028848
[AE ] epoch 45 recon_loss=0.026279
[AE ] epoch 50 recon_loss=0.021987
[AE ] epoch 55 recon_loss=0.020850
[AE ] epoch 60 recon_loss=0.018487
✅ Best AE recon_loss=0.018487
💾 Saved AE checkpoint → checkpoints/mouse_autoencoder.pt


In [9]:
# ============================================================
# 8. Export embeddings for all sessions
# ============================================================

# Reload best models
clf.load_state_dict(torch.load(best_clf_path, map_location=device))
ae.load_state_dict(torch.load(best_ae_path, map_location=device))

clf.eval()
ae.eval()

emb_cls = []
emb_ae = []

with torch.no_grad():
    for xb, yb in all_loader:
        xb = xb.to(device)
        z_cls = clf.encode(xb)   # (batch, 32)
        z_ae = ae.encode(xb)     # (batch, 16)
        emb_cls.append(z_cls.cpu().numpy())
        emb_ae.append(z_ae.cpu().numpy())

emb_cls = np.vstack(emb_cls)
emb_ae = np.vstack(emb_ae)

print("Embeddings CLS shape:", emb_cls.shape)
print("Embeddings AE shape :", emb_ae.shape)

EMB_CLS_PATH = EMB_DIR / "mouse_embeddings_cls.npy"
EMB_AE_PATH  = EMB_DIR / "mouse_embeddings_ae.npy"

np.save(EMB_CLS_PATH, emb_cls)
np.save(EMB_AE_PATH, emb_ae)

print(f"💾 Saved classifier embeddings → {EMB_CLS_PATH} shape: {emb_cls.shape}")
print(f"💾 Saved AE embeddings         → {EMB_AE_PATH} shape: {emb_ae.shape}")

Embeddings CLS shape: (1676, 32)
Embeddings AE shape : (1676, 16)
💾 Saved classifier embeddings → embeddings/mouse_embeddings_cls.npy shape: (1676, 32)
💾 Saved AE embeddings         → embeddings/mouse_embeddings_ae.npy shape: (1676, 16)


In [10]:
# ============================================================
# 9. Summary
# ============================================================

print("\n===== Mouse Encoders Summary =====")
print(f"Sessions          : {N}")
print(f"Feature dim       : {D}")
print(f"Users             : {n_users}")
print(f"Classifier emb dim: {emb_dim_cls}")
print(f"AE emb dim        : {emb_dim_ae}")
print(f"Best val accuracy : {best_val_acc:.4f}")
print(f"Best AE recon loss: {best_recon:.6f}")


===== Mouse Encoders Summary =====
Sessions          : 1676
Feature dim       : 33
Users             : 10
Classifier emb dim: 32
AE emb dim        : 16
Best val accuracy : 0.3780
Best AE recon loss: 0.018487
